# 0. Imports

In [1]:
#!pip install -qq ipython numpy pandas scikit-learn statsmodels xgboost torch

In [2]:
import sys
import warnings
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

import sklearn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import torch
from torch import nn
from torch.utils.data import Dataset, TensorDataset, DataLoader

In [3]:
!python --version
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)

Python 3.10.18
numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2
torch 2.9.1


# 1. Preprocessing

In [4]:
# AUTOREGRESSIVE (LAG) FEATURES

lags = sorted(set(
    list(range(1, 7)) + [12, 18]
    + list(range(24, 27)) + [36, 48]
    + [24*i for i in range(3,7)]
    + [24*7*i for i in range(1,5)]
))
lagFeatures = [f"Adjusted demand -{h} hr" for h in lags]

In [5]:
# CALENDAR FEATURES

# Raw integer calendar features
intDateTimeFeatures = ["Hour", "Month", "DayOfWeek", "DayOfYear"]

# Low order hour of day and day of year Fourier term features
hourFourierFeatures, dayFourierFeatures = [], []
for i in (1, 2, 3):
    argStr = (f"{i}*" if i>1 else "") + "Hour"
    hourFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])
    argStr = (f"{i}*" if i>1 else "") + "DayOfYear"
    dayFourierFeatures.extend([f"sin({argStr})", f"cos({argStr})"])

# Hour of day and day of week one-hot encodings.
# Weekend and holiday flags.
hourDummyFeatures = [f"Hour_Flag_{h}" for h in range(24)]
dayDummyFeatures = ["Day_Flag_Weekend", "Day_Flag_Holiday"]
dayDummyFeatures += [f"DayOfWeek_Flag_{d}" for d in range(7)]
monthDummyFeatures = [f"Month_Flag_{m}" for m in range(1, 13)]

# Collate all calendar features
calendarFeatures = (
    intDateTimeFeatures
    + hourFourierFeatures
    + dayFourierFeatures
    + hourDummyFeatures
    + dayDummyFeatures
    + monthDummyFeatures
)


In [6]:
# ENERGY FEATURES

energyFeatures = [
    "Adjusted net generation",
    "Adjusted total interchange",
    "FPC", "FMPP", "SOCO", "TEC",
    "JEA", "SEC", "HST", "GVL",
]


In [7]:
# WEATHER FEATURES

skyCodes = ['BKN', 'CLR', 'FEW', 'SCT', 'OVC', 'NA']
directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW", "VRB"]

weatherFeatures = ([
    "HourlyDryBulbTemperature",
    "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "HourlySeaLevelPressure",
    "HourlyVisibility",
    "HourlyWindSpeed"]
    + [f"HourlySkyConditions_Flag_{code}" for code in skyCodes]
    + [f"HourlyWindDirection_Flag_{d}" for d in directions]
    + ["sin(HourlyWindDirection)", "cos(HourlyWindDirection)"]
)


In [8]:
# Define other convenient feature variables.

target = "Adjusted demand"
allFeatures = (
    ['t', target]
    + lagFeatures 
    + calendarFeatures 
    + energyFeatures 
    + weatherFeatures
)
someFeatures = (
    ['t', target]
    + lagFeatures[:1] 
    + hourFourierFeatures[:2] + dayFourierFeatures[:2] + dayDummyFeatures[:1]
    + energyFeatures[:2] 
    + weatherFeatures[:1]
)


### 1.3 Data Splits

In [9]:
# reproducibility
np.random.seed(0)
torch.manual_seed(0)

# --- paths ---
data_root   = Path("data/clean")
train_dir   = data_root / "train"
val_dir     = data_root / "val"

DFtrain = pd.read_pickle(train_dir / "DFtrain.pkl")
DFval   = pd.read_pickle(val_dir   / "DFval.pkl")

# --- define target and features (ADJUST target name to yours) ---
TARGET_COL   = "Adjusted demand"   # <--- change to your target column
TIME_COL     = "t"        # if you have a time column
FEATURE_COLS = someFeatures[2:]

print("n_train:", len(DFtrain), "n_val:", len(DFval))
print("n_features:", len(FEATURE_COLS))


n_train: 59658 n_val: 12784
n_features: 9


# 2. LSTM

In [10]:


# Fit scalers on TRAIN ONLY
x_scaler_lstm = StandardScaler()
y_scaler_lstm = StandardScaler()

X_train_all = DFtrain[FEATURE_COLS].to_numpy()
y_train_all = DFtrain[TARGET_COL].to_numpy()

x_scaler_lstm.fit(X_train_all)
y_scaler_lstm.fit(y_train_all.reshape(-1, 1))

def df_to_scaled_arrays(df):
    """Convert a DataFrame to scaled X, y arrays for the LSTM."""
    X = df[FEATURE_COLS].to_numpy()
    y = df[TARGET_COL].to_numpy()
    X_s = x_scaler_lstm.transform(X)
    y_s = y_scaler_lstm.transform(y.reshape(-1, 1)).ravel()
    return X_s, y_s


# =========================
# 3. Sequence builders
# =========================

def make_seq(X, y, seq_len):
    """
    Build overlapping (sequence, target) pairs from one continuous block.

    X: (N, num_features), y: (N,)
    returns:
      X_seq: (N - seq_len, seq_len, num_features)
      y_seq: (N - seq_len,)
    """
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32).reshape(-1)

    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])

    # Block too short => no sequences
    if not Xs:
        return None, None

    return torch.tensor(np.stack(Xs)), torch.tensor(np.array(ys))


def make_seq_from_blocks(block_paths, seq_len):
    """
    For a list of block .pkl files, build sequences per block and concatenate.
    Skips blocks that are too short for at least one full sequence.
    """
    X_seq_list = []
    y_seq_list = []

    for path in block_paths:
        df_block = pd.read_pickle(path)

        if len(df_block) <= seq_len:
            print(f"Skipping {path.name}: length {len(df_block)} <= seq_len={seq_len}")
            continue

        X_s, y_s = df_to_scaled_arrays(df_block)
        X_block, y_block = make_seq(X_s, y_s, seq_len)

        if X_block is None:
            print(f"Skipping {path.name}: no sequences produced")
            continue

        X_seq_list.append(X_block)
        y_seq_list.append(y_block)

    if not X_seq_list:
        raise ValueError(
            f"No blocks produced sequences; ensure some blocks have length > seq_len={seq_len}."
        )

    X_seq_all = torch.cat(X_seq_list, dim=0)
    y_seq_all = torch.cat(y_seq_list, dim=0)
    return X_seq_all, y_seq_all


# =========================
# 4. Build train/val sequences & DataLoaders
# =========================

seq_len = 24  # use past 24 hours to predict next hour

# adjust patterns if your filenames differ
train_block_paths = sorted(train_dir.glob("DFtrain_block*.pkl"))
val_block_paths   = sorted(val_dir.glob("DFval_block*.pkl"))

print("Train blocks:", len(train_block_paths), " Val blocks:", len(val_block_paths))

Xtr_seq, ytr_seq = make_seq_from_blocks(train_block_paths, seq_len)
Xval_seq, yval_seq = make_seq_from_blocks(val_block_paths, seq_len)

print("Train sequences:", Xtr_seq.shape)
print("Val sequences  :", Xval_seq.shape)

train_loader_lstm = DataLoader(
    TensorDataset(Xtr_seq, ytr_seq),
    batch_size=128,
    shuffle=True,
)
val_loader_lstm = DataLoader(
    TensorDataset(Xval_seq, yval_seq),
    batch_size=128,
    shuffle=False,
)


# =========================
# 5. LSTM model definition
# =========================

class LSTMRegressor(nn.Module):
    """
    LSTM-based regressor that maps a sequence of feature vectors
    to a single scalar prediction (next-hour demand).
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch_size, seq_len, num_features)
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]   # last time step
        output = self.fc(last_hidden)      # (batch_size, 1)
        return output.squeeze(-1)          # (batch_size,)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

input_size = Xtr_seq.shape[2]


# =========================
# 6. Train + validate one LSTM config
# =========================

def train_lstm_and_eval(hidden_size, num_layers, lr=1e-3, n_epochs=10):
    """
    Train one LSTM configuration on training sequences and
    return the final validation RMSE (in original units) and the trained model.
    """
    model = LSTMRegressor(
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=0.2,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()  # MSE in scaled y-space

    for epoch in range(1, n_epochs + 1):
        # --- train epoch ---
        model.train()
        batch_losses = []

        for X_batch, y_batch in train_loader_lstm:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            y_pred_batch = model(X_batch)
            loss = criterion(y_pred_batch, y_batch)
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

        # --- validation at end of epoch ---
        model.eval()
        val_preds_scaled = []
        val_true_scaled  = []

        with torch.no_grad():
            for X_batch, y_batch in val_loader_lstm:
                X_batch = X_batch.to(device)
                y_pred = model(X_batch)
                val_preds_scaled.append(y_pred.cpu().numpy())
                val_true_scaled.append(y_batch.numpy())

        y_pred_s = np.concatenate(val_preds_scaled)
        y_true_s = np.concatenate(val_true_scaled)

        # back to original units
        y_pred = y_scaler_lstm.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
        y_true = y_scaler_lstm.inverse_transform(y_true_s.reshape(-1, 1)).ravel()

        val_rmse = root_mean_squared_error(y_true, y_pred)

        print(
            f"[LSTM hs={hidden_size}, layers={num_layers}, lr={lr}] "
            f"Epoch {epoch:02d}  Train MSE (scaled)={np.mean(batch_losses):.3f}  "
            f"Val RMSE={val_rmse:.2f}"
        )

    return val_rmse, model


# =========================
# 7. Tiny hyperparameter search on train/val
# =========================

hidden_sizes    = [32, 64]
num_layers_list = [1, 2]
learning_rates  = [1e-3]   # you can add 3e-4 if you want

best_cfg   = None
best_rmse  = np.inf
best_model = None

for hs in hidden_sizes:
    for nl in num_layers_list:
        for lr in learning_rates:
            print(f"\n=== LSTM config: hidden_size={hs}, num_layers={nl}, lr={lr} ===")
            val_rmse, model = train_lstm_and_eval(hs, nl, lr, n_epochs=10)
            if val_rmse < best_rmse:
                best_rmse  = val_rmse
                best_cfg   = dict(hidden_size=hs, num_layers=nl, lr=lr)
                best_model = model

print(f"\nBest LSTM config: {best_cfg}  (val RMSE={best_rmse:.2f})")


Train blocks: 130  Val blocks: 30
Skipping DFtrain_block010.pkl: length 23 <= seq_len=24
Skipping DFtrain_block024.pkl: length 24 <= seq_len=24
Skipping DFtrain_block038.pkl: length 5 <= seq_len=24
Skipping DFtrain_block041.pkl: length 18 <= seq_len=24
Skipping DFtrain_block051.pkl: length 2 <= seq_len=24
Skipping DFtrain_block065.pkl: length 3 <= seq_len=24
Skipping DFtrain_block067.pkl: length 17 <= seq_len=24
Skipping DFtrain_block074.pkl: length 20 <= seq_len=24
Skipping DFtrain_block075.pkl: length 20 <= seq_len=24
Skipping DFtrain_block079.pkl: length 1 <= seq_len=24
Skipping DFtrain_block080.pkl: length 1 <= seq_len=24
Skipping DFtrain_block082.pkl: length 20 <= seq_len=24
Skipping DFtrain_block083.pkl: length 20 <= seq_len=24
Skipping DFtrain_block084.pkl: length 22 <= seq_len=24
Skipping DFtrain_block099.pkl: length 2 <= seq_len=24
Skipping DFtrain_block126.pkl: length 15 <= seq_len=24
Skipping DFtrain_block129.pkl: length 1 <= seq_len=24
Skipping DFval_block007.pkl: length 1 